# Observation-level schema validation dataframe

Create one dataframe row per validation observation. Observation and candidate columns retain the exact Structured Output/Pydantic field names and values; candidate columns are empty when an observation has no candidate. `review_family` is a manually defined analysis aid derived from `proposed_name`, not a model output.

In [ ]:
import json
from pathlib import Path
from typing import Any

import pandas as pd

from schema_development.paths import ROOT


pd.set_option('display.max_colwidth', None)

In [ ]:
RESULTS_PATH = ROOT / "artifacts/validation1/results.jsonl"


In [ ]:
CANDIDATE_FAMILIES: dict[str, set[str]] = {
    "Source-document retrieval": {
        "source_document_url",
        "source_document_locator",
    },
    "Source-document identity and type": {
        "source_document_identifier",
        "source_document_type",
    },
    "Source-document publication date": {
        "source_document_publication_date",
        "source_document_date",
    },
    "Source-document attribution": {
        "source_document_creator",
        "source_document_contributor",
        "source_document_author",
        "source_document_producer",
        "source_document_organization",
        "source_document_attribution",
    },
    "Snapshot artifact date": {
        "snapshot_date",
        "snapshot_production_date",
        "snapshot_creation_date",
    },
    "Snapshot artifact attribution": {
        "snapshot_creator",
        "snapshot_producer",
        "snapshot_contributor",
        "snapshot_attribution",
    },
    "Cartographic scale and coordinates": {
        "map_scale",
        "cartographic_scale",
        "map_scale_unit",
        "coordinate_grid",
        "coordinate_extent",
    },
    "Analytical variable role": {
        "axis_variable",
        "dependent_variable",
        "explanatory_variable",
        "outcome_variable",
        "model_variable_role",
    },
    "Classification semantics": {
        "classification_system",
        "classification_level",
        "classification_rule",
        "category_hierarchy",
    },
    "Composite visualization structure": {
        "visualization_composition",
        "visualization_component_type",
        "visualization_components",
        "embedded_visualization_type",
    },
    "Data-collection timing and responsibility": {
        "data_collection_frequency",
        "data_collection_schedule",
        "data_collection_responsibility",
        "data_collection_responsible_party",
    },
    "Sample size": {"sample_size"},
    "Uncertainty representation": {"uncertainty_representation"},
    "Reference event or context": {
        "reference_event",
        "temporal_reference_event",
        "event_context",
    },
}

FAMILY_BY_CANDIDATE = {
    proposed_name: family
    for family, proposed_names in CANDIDATE_FAMILIES.items()
    for proposed_name in proposed_names
}


In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSON objects from a JSONL file.

    Parameters
    ----------
    path : Path
        JSONL file to read.

    Returns
    -------
    list[dict[str, Any]]
        Parsed records.

    Raises
    ------
    FileNotFoundError
        If the JSONL file does not exist.
    ValueError
        If a non-empty line is not valid JSON.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Results file not found: {path}")

    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}."
                ) from exc
    return records


def build_observation_dataframe(results: list[dict[str, Any]]) -> pd.DataFrame:
    """Build one dataframe row per validation observation.

    Parameters
    ----------
    results : list[dict[str, Any]]
        Successful parsed validation records.

    Returns
    -------
    pandas.DataFrame
        Observation-level table using exact structured-output field names.

    Raises
    ------
    ValueError
        If a candidate references an unknown observation, an observation supports
        multiple candidates, or an observation key is duplicated.
    """
    rows = []
    for record in results:
        parsed = record["parsed_output"]
        observations = parsed["observations"]
        observation_ids = {item["observation_id"] for item in observations}
        candidate_by_observation = {}

        for candidate in parsed["candidate_new_fields"]:
            for observation_id in candidate["supporting_observation_ids"]:
                if observation_id not in observation_ids:
                    raise ValueError(
                        f"Unknown observation {observation_id!r} in "
                        f"{record['snapshot_file_name']}."
                    )
                if observation_id in candidate_by_observation:
                    raise ValueError(
                        f"Observation {observation_id!r} supports multiple "
                        f"candidates in {record['snapshot_file_name']}."
                    )
                candidate_by_observation[observation_id] = candidate

        for observation in observations:
            candidate = candidate_by_observation.get(observation["observation_id"], {})
            rows.append(
                {
                    "snapshot_file_name": record["snapshot_file_name"],
                    "source": record["source"],
                    "artifact_type": record["artifact_type"],
                    **observation,
                    "proposed_name": candidate.get("proposed_name"),
                    "review_family": FAMILY_BY_CANDIDATE.get(
                        candidate.get("proposed_name")
                    ),
                    "definition": candidate.get("definition"),
                    "supporting_observation_ids": candidate.get(
                        "supporting_observation_ids"
                    ),
                    "why_existing_fields_are_insufficient": candidate.get(
                        "why_existing_fields_are_insufficient"
                    ),
                    "operational_value": candidate.get("operational_value"),
                    "source_level": candidate.get("source_level"),
                }
            )

    dataframe = (
        pd.DataFrame(rows)
        .sort_values(
            ["source", "artifact_type", "snapshot_file_name", "observation_id"]
        )
        .reset_index(drop=True)
    )
    key_columns = ["source", "snapshot_file_name", "observation_id"]
    if dataframe.duplicated(key_columns).any():
        raise ValueError("Observation keys must be unique.")
    return dataframe


In [ ]:
results = load_jsonl(RESULTS_PATH)
df = build_observation_dataframe(results)
df


# Review by candidate families

In [ ]:
list(CANDIDATE_FAMILIES.keys())

In [ ]:
x = "Classification semantics"
display(CANDIDATE_FAMILIES[x])
display(df[df["review_family"] == x])

# Review by unfamilied

In [ ]:
df.loc[df["proposed_name"].notna() & df["review_family"].isna()].sort_values(
    ["source", "snapshot_file_name"]
)

# Review by boundary-quality rows

In [ ]:
df.loc[df["fit_status"].isin(["uncertain", "out_of_scope"])].sort_values(
    ["source", "snapshot_file_name"]
)